In [1]:
# connect to google drive
from google.colab import drive
drive.mount('/content/gdrive')

# call all functions used in starter file
%run "/content/gdrive/MyDrive/PalmWatch/Colab Notebooks/ImportFilesAndFunctions.ipynb"

Mounted at /content/gdrive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.1 MB/s eta 0:00:00


INFO:lightning_fabric.utilities.seed:Seed set to 42


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
All imports successful!


In [2]:
# load the dataset
palmoil_df = pd.read_csv('/content/gdrive/MyDrive/PalmWatch/data/hex_deforestation_forecast.csv')

# check for any null values
palmoil_df[palmoil_df.isnull().any(axis=1)]

# fills all empty/na/nan values (usually provinces with no neighbors) with 0
palmoil_df.fillna(0, inplace=True)

In [3]:
# drop years that we need to predict
palmoil_wo_target = palmoil_df[palmoil_df['year'] <= 2018]

In [4]:
# list with features and targets
feature_names = ['defor_frac', 'degr_frac', 'remaining_frac_baseline', 'remaining_frac_hex', 'remaining_forest_ha', 'nbr_r1_defor_frac', 'nbr_r2_defor_frac' , 'nbr_r1_remaining_frac_baseline', 'target_total_5yr' , 'target_palm_5yr']

# colloquial names for every feature
common_names = ['Fraction of Deforestation', 'Fraction of Degradation', 'Cumulative Remaining Forest', 'Current Forest Density', 'Remaining Forest (hectacres)', 'Neighboring 6 Deforestation Fraction', 'Neighboring 12 Deforestation Fraction', 'Average Neighboring Remaining Forest' , 'Predicted 5 Year Deforestation', 'Predicted 5 Year Deforestation for Palm Oil']

# list of all provinces
provinces = ['Aceh', 'SumateraUtara', 'Riau', 'SumateraBarat', 'Lampung', 'SumateraSelatan', 'Jambi', 'Bengkulu']

# year range of our dataset
YEARS = range(2000, 2019)

In [5]:
# function to create a dataframe of a specific province and year
def sortByProvince(province, df, year):
  """Create a new dataframe and return all rows of a specific province and specific year"""
  new_df = df[(df['province'] == province) & (df['year'] == year)]

  return new_df

In [6]:
# function to loop through all features and find the median of a group of 16 hex id's
def create_subsetted_data(province, df, year, feature_names, group_size=16):
  """Creates a new dataframe that finds the median of a group of 16 hex_id"""
  province_df = sortByProvince(province, df, year)

  subsetted_data = {
      "Grouped ID": [],
      "Year": []
  }

  for feature in feature_names:
      subsetted_data[f"{feature}_median"] = []

  for i in range(0, len(province_df), group_size):
      subsetted_data["Grouped ID"].append(i // group_size)
      subsetted_data["Year"].append(year)

      for feature in feature_names:
          subsetted_data[f"{feature}_median"].append(
              province_df[feature].iloc[i:i+group_size].median()
          )

  return pd.DataFrame(subsetted_data)

In [7]:
# creates a dataframe with all years of a specific province
def create_all_years_data(province, df, years, feature_names, group_size=16):
  """Creates a dataframe with all years of a specific province"""
  all_data = []
  df = df.sort_values(['year', 'province', 'lat', 'lon'])

  for year in years:
      yearly_df = create_subsetted_data(
          province,
          df,
          year,
          feature_names,
          group_size
      )
      all_data.append(yearly_df)

  return pd.concat(all_data, ignore_index=True)

In [8]:
# create a new dataframe province by province of a larger grouped area

# creates a dataframe with aceh
combined_aceh = create_all_years_data(
    province="Aceh",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_aceh.insert(2, 'Province', 'Aceh')

# creates a dataframe with sumatera utara
combined_sumaterautara = create_all_years_data(
    province="SumateraUtara",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumaterautara.insert(2, 'Province', 'SumateraUtara')

# creates a dataframe with riau
combined_riau = create_all_years_data(
    province="Riau",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_riau.insert(2, 'Province', 'Riau')

# creates a dataframe with sumatera barat
combined_sumaterabarat = create_all_years_data(
    province="SumateraBarat",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumaterabarat.insert(2, 'Province', 'SumateraBarat')

# creates a dataframe with lampung
combined_lampung = create_all_years_data(
    province="Lampung",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_lampung.insert(2, 'Province', 'Lampung')

# creates a dataframe with sumatera selatan
combined_sumateraselatan = create_all_years_data(
    province="SumateraSelatan",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumateraselatan.insert(2, 'Province', 'SumateraSelatan')

# creates a dataframe with jambi
combined_jambi = create_all_years_data(
    province="Jambi",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_jambi.insert(2, 'Province', 'Jambi')

# creates a dataframe with bengkulu
combined_bengkulu = create_all_years_data(
    province="Bengkulu",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_bengkulu.insert(2, 'Province', 'Bengkulu')

In [9]:
def create_lag_features(df, features, lags=2):
  df = df.sort_values("year").copy()

  for feature in features:
      for lag in range(1, lags+1):
          df[f"{feature}_lag{lag}"] = df[feature].shift(lag)

  return df

In [10]:
aceh = palmoil_wo_target[palmoil_wo_target['province'] == 'Aceh']
aceh_18yr = aceh.groupby('year', as_index=False)[feature_names].mean()

sumaterautara = palmoil_wo_target[palmoil_wo_target['province'] == 'SumateraUtara']
sumaterautara_18yr = sumaterautara.groupby('year', as_index=False)[feature_names].mean()

riau = palmoil_wo_target[palmoil_wo_target['province'] == 'Riau']
riau_18yr = riau.groupby('year', as_index=False)[feature_names].mean()

sumaterabarat = palmoil_wo_target[palmoil_wo_target['province'] == 'SumateraBarat']
sumaterabarat_18yr = sumaterabarat.groupby('year', as_index=False)[feature_names].mean()

lampung = palmoil_wo_target[palmoil_wo_target['province'] == 'Lampung']
lampung_18yr = lampung.groupby('year', as_index=False)[feature_names].mean()

sumateraselatan = palmoil_wo_target[palmoil_wo_target['province'] == 'SumateraSelatan']
sumateraselatan_18yr = sumateraselatan.groupby('year', as_index=False)[feature_names].mean()

jambi = palmoil_wo_target[palmoil_wo_target['province'] == 'Jambi']
jambi_18yr = jambi.groupby('year', as_index=False)[feature_names].mean()

bengkulu = palmoil_wo_target[palmoil_wo_target['province'] == 'Bengkulu']
bengkulu_18yr = bengkulu.groupby('year', as_index=False)[feature_names].mean()

In [11]:
full_provinces = [aceh, sumaterautara, riau, sumaterabarat, lampung, sumateraselatan, jambi, bengkulu]

full_provinces_18yr = [aceh_18yr, sumaterautara_18yr, riau_18yr, sumaterabarat_18yr, lampung_18yr, sumateraselatan_18yr, jambi_18yr, bengkulu_18yr]

In [27]:
palm_forecast_dfs = {}
total_forecast_dfs = {}

## PALM OIL DEFORESTATION DATAFRAME

In [32]:
for province in provinces:
  print(province)
  display(palm_forecast_dfs[province])

Aceh


,Year,Palm Oil
0,2000,0.011025
1,2001,0.011115
2,2002,0.011625
3,2003,0.015949
4,2004,0.017826
5,2005,0.018112
6,2006,0.020195
7,2007,0.020443
8,2008,0.019329
9,2009,0.019280


SumateraUtara


,Year,Palm Oil
0,2000,0.068484
1,2001,0.072039
2,2002,0.074606
3,2003,0.080049
4,2004,0.082583
5,2005,0.082529
6,2006,0.088597
7,2007,0.088278
8,2008,0.087397
9,2009,0.081724


Riau


...

,Year,Palm Oil
0,2000,0.022016
1,2001,0.027570
2,2002,0.030420
3,2003,0.035029
4,2004,0.036452
5,2005,0.037314
6,2006,0.035332
7,2007,0.038606
8,2008,0.036638
9,2009,0.032303


Jambi


,Year,Palm Oil
0,2000,0.033489
1,2001,0.038954
2,2002,0.040365
3,2003,0.050173
4,2004,0.048173
5,2005,0.045990
6,2006,0.042986
7,2007,0.043708
8,2008,0.038908
9,2009,0.034164


Bengkulu


,Year,Palm Oil
0,2000,0.028364
1,2001,0.028977
2,2002,0.030091
3,2003,0.032562
4,2004,0.028812
5,2005,0.027907
6,2006,0.027798
7,2007,0.024969
8,2008,0.021941
9,2009,0.019803


In [31]:
palm_forecast_dfs['Aceh']

,Year,Palm Oil
0,2000,0.011025
1,2001,0.011115
2,2002,0.011625
3,2003,0.015949
4,2004,0.017826
5,2005,0.018112
6,2006,0.020195
7,2007,0.020443
8,2008,0.019329
9,2009,0.019280


## TOTAL DEFORESTATION DATAFRAMES

In [33]:
for province in provinces:
  print(province)
  display(total_forecast_dfs[province])

Aceh


,Year,Total Deforestation
0,2000,0.037387
1,2001,0.034874
2,2002,0.034121
3,2003,0.044577
4,2004,0.049114
5,2005,0.049545
6,2006,0.054936
7,2007,0.057943
8,2008,0.058813
9,2009,0.066768


SumateraUtara


,Year,Total Deforestation
0,2000,0.116569
1,2001,0.115971
2,2002,0.117478
3,2003,0.127660
4,2004,0.133425
5,2005,0.134941
6,2006,0.144872
7,2007,0.145103
8,2008,0.145790
9,2009,0.144657


Riau


...

,Year,Total Deforestation
0,2000,0.116629
1,2001,0.141730
2,2002,0.144697
3,2003,0.163350
4,2004,0.169075
5,2005,0.170191
6,2006,0.160154
7,2007,0.187098
8,2008,0.188439
9,2009,0.182499


Jambi


,Year,Total Deforestation
0,2000,0.095891
1,2001,0.112527
2,2002,0.118899
3,2003,0.145860
4,2004,0.141069
5,2005,0.133315
6,2006,0.123905
7,2007,0.130563
8,2008,0.125553
9,2009,0.121296


Bengkulu


,Year,Total Deforestation
0,2000,0.086071
1,2001,0.088622
2,2002,0.087038
3,2003,0.097503
4,2004,0.086116
5,2005,0.084831
6,2006,0.086352
7,2007,0.088835
8,2008,0.086653
9,2009,0.088866
